# LAL-Parser + BERTimbau para Análise de Dependência em Português

**TCC**: Desenvolvimento de um Modelo Transformer Encoder para Análise de Dependência na Língua Portuguesa

**Autor**: Leonardo Gonçalves Marotta - UFPel

Este notebook treina o LAL-Parser com BERTimbau no dataset Bosque UD.

**Requisitos**: GPU runtime no Colab (Runtime > Change runtime type > T4 GPU)

## 1. Verificar GPU

In [79]:
import torch
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('ATENCAO: Sem GPU! Va em Runtime > Change runtime type > T4 GPU')

CUDA disponível: True
GPU: Tesla T4
VRAM: 14.6 GB


## 2. Clonar o repositório

In [80]:
!git clone https://github.com/LeoMarotta/LAL-Parser-PT.git
%cd LAL-Parser-PT

Cloning into 'LAL-Parser-PT'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 152 (delta 49), reused 144 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 381.31 KiB | 2.01 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT


## 3. Instalar dependências

In [81]:
!pip install -q torch numpy cython nltk tqdm transformers sentencepiece
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

## 4. Compilar EVALB

In [82]:
!cd EVALB && make

gcc -Wall -g -o evalb evalb.c
evalb.c: In function ‘main’:
evalb.c:378:22: warning: pointer targets in passing argument 1 of ‘fgets’ differ in signedness []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wpointer-sign-Wpointer-sign]8;;]
  378 |     for(Line=1;fgets(buff,5000,fd1)!=NULL;Line++){
      |                      ^~~~
      |                      |
      |                      unsigned char *
In file included from evalb.c:21:
/usr/include/stdio.h:592:38: note: expected ‘char * restrict’ but argument is of type ‘unsigned char *’
  592 | extern char *fgets (char *__restrict __s, int __n, FILE *__restrict __stream)
      |                     ~~~~~~~~~~~~~~~~~^~~
evalb.c:385:16: warning: pointer targets in passing argument 1 of ‘strcpy’ differ in signedness []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wpointer-sign-Wpointer-sign]8;;]
  385 |         strcpy(buff1,buff);
      |                ^~~~~
      |                |
      |   

## 5. Preparar dataset Bosque UD

In [83]:
# Sobrescrevendo o script com lógica rigorosa de alinhamento 1:1
import os

script_content = """
import os
import requests

def sanitize(text):
    return text.replace(' ', '_') if text else text

def download_file(url, dest):
    print(f'  Baixando {url}...')
    r = requests.get(url)
    with open(dest, 'wb') as f:\n        f.write(r.content)

def convert(input_path, out_const, out_dep):
    sentences = []
    with open(input_path, 'r', encoding='utf-8') as f:
        curr = []
        for line in f:
            if line.startswith('#'): continue
            if line.strip() == '':
                if curr: sentences.append(curr)
                curr = []
            else:
                cols = line.strip().split('\\t')
                # Ignora tokens multi-word (ex: 1-2) e tokens vazios (ex: 1.1)
                if '-' in cols[0] or '.' in cols[0]: continue
                curr.append(cols)

    with open(out_const, 'w', encoding='utf-8') as f_c, open(out_dep, 'w', encoding='utf-8') as f_d:
        for sent in sentences:
            words = [sanitize(cols[1]) for cols in sent]
            tags = [cols[3] for cols in sent]

            # Gera constituency linear (TOP (S (TAG WORD)...))
            const_str = '(TOP (S ' + ' '.join([f'({t} {w})' for t, w in zip(tags, words)]) + '))'
            f_c.write(const_str + '\\n')

            # Gera dependency (CoNLL-X) com IDs resetados para garantir 1, 2, 3...
            for i, cols in enumerate(sent, 1):
                cols[0] = str(i)
                cols[1] = sanitize(cols[1])
                f_d.write('\\t'.join(cols) + '\\n')
            f_d.write('\\n')
    return len(sentences)

# Setup
os.makedirs('data/bosque', exist_ok=True)
urls = {
    'train': 'https://raw.githubusercontent.com/UniversalDependencies/UD_Portuguese-Bosque/master/pt_bosque-ud-train.conllu',
    'dev': 'https://raw.githubusercontent.com/UniversalDependencies/UD_Portuguese-Bosque/master/pt_bosque-ud-dev.conllu',
    'test': 'https://raw.githubusercontent.com/UniversalDependencies/UD_Portuguese-Bosque/master/pt_bosque-ud-test.conllu'
}

for split, url in urls.items():
    raw_path = f'data/bosque/{split}.conllu'
    download_file(url, raw_path)
    count = convert(raw_path, f'data/bosque/bosque_{split}.clean', f'data/bosque/bosque_{split}.conllx')
    print(f'  {split}: {count} sentenças processadas com alinhamento 1:1.')
"""

with open('scripts/prepare_bosque.py', 'w') as f:
    f.write(script_content)

!python scripts/prepare_bosque.py

  Baixando https://raw.githubusercontent.com/UniversalDependencies/UD_Portuguese-Bosque/master/pt_bosque-ud-train.conllu...
  train: 7018 sentenças processadas com alinhamento 1:1.
  Baixando https://raw.githubusercontent.com/UniversalDependencies/UD_Portuguese-Bosque/master/pt_bosque-ud-dev.conllu...
  dev: 1172 sentenças processadas com alinhamento 1:1.
  Baixando https://raw.githubusercontent.com/UniversalDependencies/UD_Portuguese-Bosque/master/pt_bosque-ud-test.conllu...
  test: 1167 sentenças processadas com alinhamento 1:1.


## 6. Verificar dados

In [84]:
!echo '=== Constituency (primeiras 3 linhas) ===' && head -3 /content/LAL-Parser-PT/data/bosque/bosque_train.clean
!echo ''
!echo '=== Dependency (primeiras 15 linhas) ===' && head -15 /content/LAL-Parser-PT/data/bosque/bosque_train.conllx
!echo ''
!echo '=== Contagem de sentenças (deve ser igual) ==='
!wc -l /content/LAL-Parser-PT/data/bosque/bosque_train.clean
!grep -c '^$' /content/LAL-Parser-PT/data/bosque/bosque_train.conllx

=== Constituency (primeiras 3 linhas) ===
(TOP (S (X (PROPN PT) (ADP em) (DET o) (NOUN governo))))
(TOP (S (X (PROPN BRASÍLIA) (PROPN Pesquisa) (PROPN Datafolha) (VERB publicada) (ADV hoje) (VERB revela) (DET um) (NOUN dado) (ADJ supreendente) (PUNCT :) (VERB recusando) (DET uma) (NOUN postura) (ADJ radical) (PUNCT ,) (DET a) (ADJ esmagadora) (NOUN maioria) (PUNCT -LRB-) (NUM 77) (SYM %) (PUNCT -RRB-) (ADP de) (DET os) (NOUN eleitores) (VERB quer) (DET o) (PROPN PT) (VERB participando) (ADP de) (DET o) (NOUN Governo) (PROPN Fernando) (PROPN Henrique) (PROPN Cardoso) (PUNCT .))))
(TOP (S (X (VERB Tem) (NOUN sentido) (PUNCT --) (ADV aliás) (PUNCT ,) (DET muitíssimo) (NOUN sentido) (PUNCT .))))

=== Dependency (primeiras 15 linhas) ===
1	PT	_	PROPN	PROPN	_	0	root	_	_
2	em	_	ADP	ADP	_	4	case	_	_
3	o	_	DET	DET	_	4	det	_	_
4	governo	_	NOUN	NOUN	_	1	nmod	_	_

1	BRASÍLIA	_	PROPN	PROPN	_	6	parataxis	_	_
2	Pesquisa	_	PROPN	PROPN	_	6	nsubj	_	_
3	Datafolha	_	PROPN	PROPN	_	2	flat:name	_	_
4	publica

## 7. Treinar o modelo

**Tempo estimado**: ~2-6 horas no Colab com T4 (depende das épocas).

Ajuste `--epochs` e `--batch-size` conforme necessário.

Se o Colab desconectar, reconecte e re-execute a partir da célula 2.

In [85]:
!mkdir -p models

!python src_joint/main.py train \
    --model-path-base models/lal_bertimbau_bosque \
    --epochs 50 \
    --use-bert \
    --bert-model "neuralmind/bert-base-portuguese-cased" \
    --no-bert-do-lower-case \
    --use-tags \
    --const-lada 0.5 \
    --dataset ptb \
    --embedding-path /content/LAL-Parser-PT/data/glove.gz \
    --model-name lal_bertimbau_bosque \
    --checks-per-epoch 4 \
    --num-layers 2 \
    --learning-rate 0.00005 \
    --batch-size 32 \
    --eval-batch-size 16 \
    --subbatch-max-tokens 500 \
    --train-ptb-path /content/LAL-Parser-PT/data/bosque/bosque_train.clean \
    --dev-ptb-path /content/LAL-Parser-PT/data/bosque/bosque_dev.clean \
    --dep-train-ptb-path /content/LAL-Parser-PT/data/bosque/bosque_train.conllx \
    --dep-dev-ptb-path /content/LAL-Parser-PT/data/bosque/bosque_dev.conllx \
    --lal-d-kv 128 \
    --lal-d-proj 128 \
    --no-lal-resdrop

/usr/local/lib/python3.12/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/src_joint/hpsg_decoder.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
In file included from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /root/.pyxbld/temp.linux-x86_64-cpython-312/content/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/src_joint/hpsg_decoder.c:1240:
/usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, 

In [86]:
import os
print(f'Diretório atual: {os.getcwd()}')
paths = [
    'data/bosque/bosque_train.clean',
    'data/bosque/bosque_train.conllx',
    'LAL-Parser-PT/data/bosque/bosque_train.clean',
    '/content/LAL-Parser-PT/data/bosque/bosque_train.clean'
]
for p in paths:
    print(f'{p}: {"Existe" if os.path.exists(p) else "Não encontrado"}')

Diretório atual: /content/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT/LAL-Parser-PT
data/bosque/bosque_train.clean: Existe
data/bosque/bosque_train.conllx: Existe
LAL-Parser-PT/data/bosque/bosque_train.clean: Não encontrado
/content/LAL-Parser-PT/data/bosque/bosque_train.clean: Existe


## 8. Avaliar no conjunto de teste

In [87]:
import glob

model_files = sorted(glob.glob('models/lal_bertimbau_bosque_dev=*'), key=lambda x: float(x.split('dev=')[1].replace('.pt', '')))
if model_files:
    best_model = model_files[-1]
    print(f'Melhor modelo: {best_model}')
else:
    print('Nenhum modelo encontrado! Treine primeiro.')
    best_model = None

Nenhum modelo encontrado! Treine primeiro.


In [88]:
if best_model:
    import os
    os.system(f'python src_joint/main.py test --dataset ptb --eval-batch-size 8 --consttest-ptb-path data/bosque/bosque_test.clean --deptest-ptb-path data/bosque/bosque_test.conllx --embedding-path data/glove.gz --model-path-base {best_model}')

## 9. Salvar modelo no Google Drive (opcional)

Para não perder o modelo quando o Colab desconectar.

In [89]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/TCC_Models
!cp models/lal_bertimbau_bosque_dev=* /content/drive/MyDrive/TCC_Models/
print('Modelos salvos no Google Drive!')

MessageError: Error: credential propagation was unsuccessful